## Azure ML Data Stores and Data Sets

### Set up the Azure ML Client

In [ ]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

### Connect to your Azure ML workspace

In [ ]:
import azureml.core
from azureml.core import Workspace

# Load the workspace from the saved config file
ws = Workspace.from_config()
print('Ready to use Azure ML {} to work with {}'.format(azureml.core.VERSION, ws.name))

### View all Default Datastores in your workspace

In [ ]:
# Get the default datastore
default_ds = ws.get_default_datastore()

# Enumerate all datastores, indicating which is the default
for ds_name in ws.datastores:
    print(ds_name, "- Default =", ds_name == default_ds.name)

### Connect a new Azure Blob Storage account as a Datastore

In [ ]:
aml_datastore = AzureBlobDatastore(
    name="aml_datastore",
    description="Datastore pointing to a blob container using https protocol.",
    account_name="YOUR-STORAGE-ACCOUNT-NAME",
    container_name="YOUR-CONTAINER-NAME",
    protocol="https",
    credentials=AccountKeyConfiguration(
        account_key="YOUR-ACCOUNT-KEY"
    ),
)

ml_client.create_or_update(aml_datastore)

### Setting up the Default Datastore

In [ ]:
from azureml.core import Datastore

# Get the datastore to use in the rest of the notebook
aml_datastore = Datastore.get(ws, 'aml_datastore')
print(aml_datastore.name,":", aml_datastore.datastore_type + " (" + aml_datastore.account_name + ")")

In [ ]:
# Set the default datastore to the one we just created
ws.set_default_datastore('aml_datastore')
default_ds = ws.get_default_datastore()
print(default_ds.name)

In [ ]:
# Get the default datastore
default_ds = ws.get_default_datastore()

# Enumerate all datastores, indicating which is the default
for ds_name in ws.datastores:
    print(ds_name, "- Default =", ds_name == default_ds.name)

### Upload data to your Datastore and create a Dataset

In [ ]:
import requests

url = "https://raw.githubusercontent.com/kuljotSB/AI-300/main/Data/diabetes.csv"

r = requests.get(url)

files = ["diabetes.csv"]
for file in files:
    with open(file, "wb") as f:
        f.write(r.content)

    default_ds.upload_files(
        files=[file],
        target_path="diabetes-data/",
        overwrite=True,
        show_progress=True
    )

### Create a Tabular Dataset from the uploaded data

In [ ]:
from azureml.core import Dataset

# Get the default datastore
default_ds = ws.get_default_datastore()

#Create a tabular dataset from the path on the datastore (this may take a short while)
tab_data_set = Dataset.Tabular.from_delimited_files(path=(default_ds, 'diabetes-data/*.csv'))

# Display the first 20 rows as a Pandas dataframe
tab_data_set.take(20).to_pandas_dataframe()

### Create a File Dataset from the uploaded data

In [ ]:
#Create a file dataset from the path on the datastore (this may take a short while)
file_data_set = Dataset.File.from_files(path=(default_ds, 'diabetes-data/*.csv'))

# Get the files in the dataset
for file_path in file_data_set.to_path():
    print(file_path)

### Register the Dataset in the workspace for later use

In [ ]:
# Register the tabular dataset
try:
    tab_data_set = tab_data_set.register(workspace=ws, 
                                        name='diabetes dataset',
                                        description='diabetes data',
                                        tags = {'format':'CSV'},
                                        create_new_version=True)
except Exception as ex:
    print(ex)

# Register the file dataset
try:
    file_data_set = file_data_set.register(workspace=ws,
                                            name='diabetes file dataset',
                                            description='diabetes files',
                                            tags = {'format':'CSV'},
                                            create_new_version=True)
except Exception as ex:
    print(ex)

print('Datasets registered')

In [ ]:
print("Datasets:")
for dataset_name in list(ws.datasets.keys()):
    dataset = Dataset.get_by_name(ws, dataset_name)
    print("\t", dataset.name, 'version', dataset.version)